# 证据链 B：工单时间和版本分开检查
CPU 演示；内含 B 的原代码和 PR #4 评分代码，不调用模型。

In [ ]:
import base64,io,json,zipfile,tempfile,subprocess,sys
from pathlib import Path
folder=Path(tempfile.mkdtemp())
with zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))) as z:
    z.extractall(folder)
folder=folder/'modules/evidence_b_checker'

## 看四轮审核及评分
第四轮收到了工单，但版本还未获独立确认。

In [ ]:
def run_file(*args):
    r=subprocess.run([sys.executable,*args],cwd=folder,capture_output=True,text=True)
    print(r.stdout,r.stderr)
    r.check_returncode()
run_file('run_dynamic_review.py')
run_file('run_score_bridge_v02.py')

## 查看第四轮原因
第一项显示工单时间的计算结果，第二项显示能否采纳这条证据。

In [ ]:
stages=json.loads((folder/'examples/dynamic_trace.json').read_text())['stages']
g=stages[-1]['snapshot']
print('只看时间：',g['checks'][0]['time_comparison'])
print('工单证据状态：',g['nodes'][-1]['state'])
print('原因：',g['gaps'][-1]['reason'])

## 运行动态审核与工单版本的回归测试

In [ ]:
run_file('-m','unittest','test_dynamic_review','test_score_binding')